# Narrow query + semantic classifier: file-held-out experiment

Run from project/ on TIP. Uses the existing 05 scores and embeddings, but trains a
NEW model. Does not use 07 predictions or alter previous outputs.
The exact attached CSV is frozen in data/experiments/narrow_holdout/test.csv.
All 100 test family IDs (including unsure rows) are removed from every training
source BEFORE fitting, scaling, cross-validation, C or threshold selection.
Test targets are the raw CSV 1/0 labels; ? is excluded, with no note-based relabelling.

C is selected by training-only OOF AUC. Threshold is selected by training-only F1
for the narrow-query-plus-classifier policy on all eligible training families,
including those missed by narrow retrieval. Test labels are not used for tuning.
The classifier uses the remaining broad training pool (not just narrow members).

This is a retrospective file-held-out evaluation: this file's results influenced
previous development and its AI labels are provisional. It is not a pristine
external benchmark. No population claim should be made from these 84 labels.
Rollback branch: experiment/narrow-query-classifier, commit 2131701.


In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from epo.tipdata.patstat import PatstatClient
from src.labels import load_labels
from src.policy_metrics import binary_metrics
from src.trial_snapshot.sector_config import load_sector_config
from src.trial_snapshot.query_builders import SectorQueryBuilder

ENV = "PROD"
ROOT = Path("outputs/step0")
DEST = ROOT / f"narrow_file_holdout_{ENV}"
DEST.mkdir(parents=True, exist_ok=True)
TEST_PATH = Path("data/experiments/narrow_holdout/test.csv")
EXPECTED_SHA = "20ef0c0af5dc1bf3d057b81a82ce1c5a8d32a75ca1f3509eb2b6fcc904a13630"
assert hashlib.sha256(TEST_PATH.read_bytes()).hexdigest() == EXPECTED_SHA, "Test snapshot changed"
SCORE_COLS = ["sim_definition", "margin", "bm25"]
C_GRID = [0.03, 0.1, 0.3, 1.0]
THRESHOLDS = np.round(np.arange(0.10, 0.91, 0.05), 2)
SEED = 42

def read_output(stem):
    p = ROOT / f"{stem}_{ENV}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(p.with_suffix(".csv"))

raw = pd.read_csv(TEST_PATH, encoding="utf-8-sig", dtype={"relevance_label": str})
assert raw["docdb_family_id"].notna().all() and not raw["docdb_family_id"].duplicated().any()
test_ids = set(raw["docdb_family_id"].astype(int))
raw["relevance"] = raw["relevance_label"].str.strip().map({"1": 1, "0": 0})
test = raw.dropna(subset=["relevance"]).copy()
all_labels = load_labels(Path("data/labels"))
train_labels = all_labels[~all_labels["docdb_family_id"].isin(test_ids)].dropna(subset=["relevance"]).copy()
assert not set(train_labels["docdb_family_id"]) & test_ids
print(f"Frozen test: {len(raw)} families; {len(test)} labelled; {len(raw)-len(test)} unsure excluded")
print(f"Training labels after removing ALL test IDs: {len(train_labels)}")


In [ ]:
# Features come only from frozen text embeddings and 05 semantic/BM25 scores.
scores = read_output("scores")
assert not scores["docdb_family_id"].duplicated().any()
wanted = set(train_labels["docdb_family_id"]) | test_ids
vectors = {}
for path in sorted((ROOT / f"embeddings_{ENV}").glob("part_*_ids.npy")):
    ids = np.load(path)
    keep = np.isin(ids, list(wanted))
    if keep.any():
        vec = np.load(str(path).replace("_ids.npy", "_vec.npy"))
        vectors.update(zip(ids[keep].tolist(), vec[keep]))
feature_table = scores[["docdb_family_id", *SCORE_COLS]].copy()
feature_table = feature_table[np.isfinite(feature_table[SCORE_COLS].to_numpy(dtype=float)).all(axis=1)]
feature_table = feature_table[feature_table["docdb_family_id"].isin(vectors)].set_index("docdb_family_id")
train = train_labels[train_labels["docdb_family_id"].isin(feature_table.index)].copy()
train_labels[~train_labels["docdb_family_id"].isin(feature_table.index)].to_csv(DEST / "training_missing_features.csv", index=False)
assert not set(train["docdb_family_id"]) & test_ids

def features(frame):
    ids = frame["docdb_family_id"].tolist()
    return np.hstack([np.vstack([vectors[i] for i in ids]), feature_table.loc[ids, SCORE_COLS].to_numpy(dtype=float)])

X = features(train)
y = train["relevance"].astype(int).to_numpy()
if len(np.unique(y)) != 2 or np.bincount(y).min() < 5:
    raise ValueError("Need at least five training families in each class")
print(f"Training with features: {len(train)}; positives: {y.sum()}; features: {X.shape[1]}")


In [ ]:
# Exact pinned trial_branch query; no approximation from representative text.
patstat = PatstatClient(env=ENV)
db = patstat.orm()
cfg = load_sector_config("src/trial_snapshot/sector.json")
sq = SectorQueryBuilder(db, cfg).family_membership()
# Restrict downloaded membership to experiment IDs; rules still match all family members.
membership = patstat.df(db.query(sq).filter(sq.c.docdb_family_id.in_([int(i) for i in wanted])))
narrow_ids = set(membership["docdb_family_id"])
membership.to_csv(DEST / "membership.csv", index=False)
train_narrow = train["docdb_family_id"].isin(narrow_ids).to_numpy()
test_narrow = test["docdb_family_id"].isin(narrow_ids)
# Unscored narrow test families cannot be silently omitted from the test denominator.
missing = test[test_narrow & ~test["docdb_family_id"].isin(feature_table.index)]
missing.to_csv(DEST / "test_missing_features.csv", index=False)
if len(missing):
    # Complete test features without changing the original 05 stores or training data.
    import re
    from collections import Counter
    from rank_bm25 import BM25Okapi
    from sentence_transformers import SentenceTransformer
    from src.sector import load_config, family_english_texts
    print(f"Completing features for {len(missing)} narrow test families", flush=True)
    semantic_cfg = load_config()
    sem = semantic_cfg["semantic"]
    encoder = SentenceTransformer(sem["model"])
    encoder.max_seq_length = sem["max_seq_length"]
    recovered = patstat.df(family_english_texts(db, missing["docdb_family_id"].astype(int).tolist()))
    # Prefer PATSTAT English text, then an explicitly English abstract in the frozen CSV.
    if recovered.empty:
        recovered = pd.DataFrame(columns=["docdb_family_id", "appln_title", "appln_abstract"])
    recovered = recovered[recovered["appln_abstract"].fillna("").str.strip().ne("")].copy()
    recovered["text_source"] = "PATSTAT English abstract"
    fallback = missing[
        ~missing["docdb_family_id"].isin(recovered["docdb_family_id"])
        & missing["appln_abstract_lg"].fillna("").str.strip().str.lower().eq("en")
        & missing["appln_abstract"].fillna("").str.strip().ne("")
    ][["docdb_family_id", "appln_title", "appln_abstract"]].copy()
    fallback["text_source"] = "Frozen test CSV English abstract"
    recovered = pd.concat([recovered, fallback], ignore_index=True)
    unavailable = missing[~missing["docdb_family_id"].isin(recovered["docdb_family_id"])].copy()
    unavailable.to_csv(DEST / "test_without_english_text.csv", index=False)
    print(f"Recoverable English texts: {len(recovered)}; pending manual review: {len(unavailable)}", flush=True)
    # No-English families stay in the test denominator and are not automatically accepted.
    # This is an operational coverage failure, not a classifier negative prediction.
    if len(recovered):
        text = recovered["appln_title"].fillna("") + ". " + recovered["appln_abstract"].fillna("")
        E = encoder.encode(text.tolist(), normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
        definition = encoder.encode(semantic_cfg["description"], normalize_embeddings=True, convert_to_numpy=True)
        negatives = encoder.encode(list(sem["negatives"].values()), normalize_embeddings=True, convert_to_numpy=True)
        # Reconstruct the ORIGINAL corpus; do not fit IDF on the two test documents.
        texts = read_output("texts")
        corpus = texts[texts["has_abstract"]].sort_values("docdb_family_id").copy()
        token = re.compile(r"[a-z0-9]+")
        tokenize = lambda value: token.findall(value.lower())
        corpus_text = corpus["appln_title"].fillna("") + ". " + corpus["appln_abstract"].fillna("")
        bm = BM25Okapi([tokenize(t) for t in corpus_text])
        query = tokenize(semantic_cfg["description"])
        rebuilt = pd.Series(bm.get_scores(query), index=corpus["docdb_family_id"])
        original = scores.set_index("docdb_family_id")["bm25"]
        common = rebuilt.index.intersection(original.index)
        if not len(common) or not np.allclose(rebuilt.loc[common], original.loc[common], rtol=1e-5, atol=1e-5):
            raise RuntimeError("Current texts/config do not reproduce stored BM25 scores. Align 05 artifacts before completing test features.")
        def frozen_bm25(value):
            tokens = tokenize(value)
            counts = Counter(tokens)
            length = len(tokens)
            return sum(bm.idf.get(q, 0.0) * counts[q] * (bm.k1 + 1) /
                       (counts[q] + bm.k1 * (1 - bm.b + bm.b * length / bm.avgdl))
                       for q in query)
        extra = pd.DataFrame({"docdb_family_id": recovered["docdb_family_id"].to_numpy(),
                              "sim_definition": E @ definition,
                              "margin": E @ definition - (E @ negatives.T).max(axis=1),
                              "bm25": [frozen_bm25(t) for t in text]})
        if E.shape[1] + len(SCORE_COLS) != X.shape[1] or not np.isfinite(extra[SCORE_COLS].to_numpy()).all():
            raise RuntimeError("Recovered test features are incompatible with training features")
        for family_id, vector in zip(recovered["docdb_family_id"], E):
            vectors[int(family_id)] = vector
        feature_table = pd.concat([feature_table, extra.set_index("docdb_family_id")])
        feature_table = feature_table[~feature_table.index.duplicated(keep="last")]
        extra.to_csv(DEST / "recovered_test_features.csv", index=False)
        recovered.to_csv(DEST / "recovered_test_texts.csv", index=False)
        np.save(DEST / "recovered_test_vectors.npy", E)
        print("Missing test features completed; original 05 artifacts unchanged", flush=True)


In [ ]:
# Select BOTH C and threshold exclusively on training OOF predictions.
def model(c):
    return make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=5000, class_weight="balanced"))
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof = {c: cross_val_predict(model(c), X, y, cv=folds, method="predict_proba")[:, 1] for c in C_GRID}
aucs = {c: roc_auc_score(y, p) for c, p in oof.items()}
best_c = max(C_GRID, key=lambda c: aucs[c])
tuning = pd.DataFrame([{"threshold": float(t), **binary_metrics(y, train_narrow & (oof[best_c] >= t))} for t in THRESHOLDS])
threshold = float(tuning.loc[tuning["f1"].idxmax(), "threshold"])
tuning.to_csv(DEST / "training_threshold_selection.csv", index=False)
train.assign(p_oof=oof[best_c], in_narrow_query=train_narrow).to_csv(DEST / "training_oof.csv", index=False)
print("Training-only AUC:", aucs)
print(f"LOCKED from training only: C={best_c}, threshold={threshold}")
final_model = model(best_c).fit(X, y)


In [ ]:
# Test is evaluated ONCE at the training-selected threshold.
result = test.copy()
result["in_narrow_query"] = result["docdb_family_id"].isin(narrow_ids)
result["p_holdout"] = np.nan
need_score = result["in_narrow_query"] & result["docdb_family_id"].isin(feature_table.index)
if need_score.any():
    result.loc[need_score, "p_holdout"] = final_model.predict_proba(features(result[need_score]))[:, 1]
result["decision_status"] = np.select(
    [~result["in_narrow_query"], result["p_holdout"].isna(), result["p_holdout"].ge(threshold)],
    ["outside_narrow_query", "pending_no_features", "accepted"], default="rejected_by_classifier")
result["narrow_plus_classifier"] = result["in_narrow_query"] & result["p_holdout"].ge(threshold)
truth = result["relevance"].astype(int)
report = pd.DataFrame([
    {"policy": "narrow_query_only", **binary_metrics(truth, result["in_narrow_query"])},
    {"policy": "narrow_plus_classifier", **binary_metrics(truth, result["narrow_plus_classifier"])},
])
print("FILE-HELD-OUT TEST: raw CSV labels, same denominator for both policies")
print(report.round(3).to_string(index=False))
print("Positives missed by narrow retrieval:", int((truth.eq(1) & ~result["in_narrow_query"]).sum()))
print("Positives rejected by classifier:", int((truth.eq(1) & result["decision_status"].eq("rejected_by_classifier")).sum()))
print("Pending without English features (not auto-accepted; included in denominator):")
print(result.loc[result["decision_status"].eq("pending_no_features"), ["docdb_family_id", "relevance", "appln_title"]].to_string(index=False))
print("Negatives removed by classifier:", int((truth.eq(0) & result["decision_status"].eq("rejected_by_classifier")).sum()))
subset = result[result["in_narrow_query"] & result["p_holdout"].notna()]
auc = float(roc_auc_score(subset["relevance"], subset["p_holdout"])) if subset["relevance"].nunique() == 2 else None
print("Classifier AUC within narrow test candidates:", auc)
report.to_csv(DEST / "test_metrics.csv", index=False)
result.to_csv(DEST / "test_predictions.csv", index=False)
metadata = {"test_sha256": EXPECTED_SHA, "test_total": len(raw), "test_labelled": len(test),
            "training_n": len(train), "test_training_overlap": 0, "C": best_c, "threshold": threshold,
            "training_auc_by_C": aucs, "test_narrow_auc": auc,
            "query_commit": "c509b9b5e7de13762c832c47ea3d0ee5019b2802",
            "pending_without_features": int(result["decision_status"].eq("pending_no_features").sum()),
            "missing_feature_policy": "manual review; not auto-accepted; retained in operational test denominator",
            "test_labels": "raw CSV 1/0; unsure excluded; no note-based remapping",
            "limitation": "Retrospective holdout; provisional AI labels; prior development saw this dataset"}
(DEST / "metadata.json").write_text(json.dumps(metadata, indent=2))
print("Saved to", DEST)
